# 日本語音声の感情分析（Before評価）
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nagomi-tech/blog-aibeginner/blob/main/Colab/japanese_speech_emotion_before.ipynb)

英語のMSP-Podcastデータで学習された `audeering/wav2vec2-large-robust-12-ft-emotion-msp-dim` を、
**ファインチューニングなし**で日本語音声にそのまま適用し、Valence（感情価）・Arousal（覚醒度）・Dominance（支配性）の3次元スコアを出力します。

この結果が今回の実験の「Before」（ベースライン）になります。

## このノートブックでできること
1. モデルのロード
2. 日本語音声ファイルのアップロード（またはgTTSでのクイック生成）
3. 各音声に対するVADスコアの推論
4. 結果の一覧表示・可視化（VAD空間へのプロット）
5. 結果をCSVで保存（Afterとの比較用）

## 想定ランタイム
- 無料版のT4 GPUで動作する想定です（モデルは12層に軽量化済み・約0.3B程度）
- メモリ不足になった場合は「ランタイム→ランタイムのタイプを変更」から有料プランのGPU（A100等）に切り替えてください

## JVNV等の本格的なデータセットを使う場合
JVNVは [IEEE DataPort](https://ieee-dataport.org/documents/jvnv-corpus-japanese-emotional-speech-verbal-content-and-nonverbal-expressions) での会員登録・申請が必要なため、このノートブックでは自動ダウンロードしていません。
ダウンロード後、zipファイルをこのノートブックの「② 音声ファイルの準備」セクションからアップロードしてください。

## ① 環境セットアップ

In [ ]:
# 必要なライブラリをインストール
!pip install -q transformers torch torchaudio librosa soundfile gTTS pandas matplotlib japanize-matplotlib

In [ ]:
import torch

print("CUDA利用可能:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPUが割り当てられていません。「ランタイム→ランタイムのタイプを変更」でGPUを選択してください。")

## ② 音声ファイルの準備

以下の2つの方法のどちらかで音声を用意してください。

- **方法A**：手元の音声ファイル（wav/mp3）をアップロードする（JVNV等の本格データセットを使う場合はこちら）
- **方法B**：gTTSで簡易的に日本語の感情テキストを音声合成し、動作確認用に使う（お試し・疎通確認向け）

どちらか一方、または両方を実行してください。

### 方法A: 音声ファイルをアップロード

In [ ]:
import os

AUDIO_DIR = "/content/audio_samples"
os.makedirs(AUDIO_DIR, exist_ok=True)

from google.colab import files

print("wav/mp3ファイルを選択してアップロードしてください（複数選択可）")
uploaded = files.upload()

for fname, content in uploaded.items():
    dest = os.path.join(AUDIO_DIR, fname)
    with open(dest, "wb") as f:
        f.write(content)
    print(f"保存しました: {dest}")

### 方法B: gTTSで簡易日本語音声を生成（お試し用）

同じテキストに近い内容を、文面のトーンだけ変えて読み上げます。
gTTSは抑揚を細かく制御できないため、**あくまで疎通確認・動作テスト用**です。
本格的な感情バリエーションを作る場合はVOICEVOX等に差し替えてください。

In [ ]:
from gtts import gTTS
import os

AUDIO_DIR = "/content/audio_samples"
os.makedirs(AUDIO_DIR, exist_ok=True)

# 文面のトーンで感情の違いを表現したサンプルテキスト
sample_texts = {
    "neutral_01": "承知いたしました。対応いたします。",
    "happy_01": "わあ、本当にありがとうございます！とても嬉しいです！",
    "angry_01": "何度も同じことを言わせないでください。いい加減にしてください。",
    "sad_01": "そうですか…もうどうしようもないんですね…",
    "fear_01": "え、それは本当ですか？どうしよう、どうすればいいか分かりません。",
}

for name, text in sample_texts.items():
    tts = gTTS(text=text, lang="ja")
    path = os.path.join(AUDIO_DIR, f"{name}.mp3")
    tts.save(path)
    print(f"生成しました: {path}  (text: {text})")

## ③ モデルのロード

`audeering/wav2vec2-large-robust-12-ft-emotion-msp-dim` は独自のモデルクラスを使うため、
公式のサンプルコードに沿ってカスタムクラスを定義してからロードします。

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from transformers import Wav2Vec2Processor
from transformers.models.wav2vec2.modeling_wav2vec2 import (
    Wav2Vec2Model,
    Wav2Vec2PreTrainedModel,
)


class RegressionHead(nn.Module):
    """VAD(Valence, Arousal, Dominance)を出力する回帰ヘッド"""

    def __init__(self, config):
        super().__init__()
        self.dense = nn.Linear(config.hidden_size, config.hidden_size)
        self.dropout = nn.Dropout(config.final_dropout)
        self.out_proj = nn.Linear(config.hidden_size, config.num_labels)

    def forward(self, features, **kwargs):
        x = features
        x = self.dropout(x)
        x = self.dense(x)
        x = torch.tanh(x)
        x = self.dropout(x)
        x = self.out_proj(x)
        return x


class EmotionModel(Wav2Vec2PreTrainedModel):
    """音声感情分類器（VAD回帰）"""

    def __init__(self, config):
        super().__init__(config)
        self.config = config
        self.wav2vec2 = Wav2Vec2Model(config)
        self.classifier = RegressionHead(config)
        self.init_weights()

    def forward(self, input_values):
        outputs = self.wav2vec2(input_values)
        hidden_states = outputs[0]
        hidden_states = torch.mean(hidden_states, dim=1)
        logits = self.classifier(hidden_states)
        return hidden_states, logits


MODEL_NAME = "audeering/wav2vec2-large-robust-12-ft-emotion-msp-dim"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"使用デバイス: {device}")

processor = Wav2Vec2Processor.from_pretrained(MODEL_NAME)
model = EmotionModel.from_pretrained(MODEL_NAME).to(device)
model.eval()

print("モデルのロードが完了しました。")

## ④ 推論関数の定義

音声ファイルを16kHz・モノラルに変換したうえでモデルに入力し、VADスコアを取得します。

In [ ]:
import librosa

TARGET_SR = 16000


def load_audio(path, target_sr=TARGET_SR):
    """音声を読み込み、16kHz・モノラルのnumpy配列に変換する"""
    audio, sr = librosa.load(path, sr=target_sr, mono=True)
    return audio


@torch.no_grad()
def predict_vad(audio_path, model, processor, device):
    """1つの音声ファイルからVAD(Valence, Arousal, Dominance)を推論する"""
    audio = load_audio(audio_path)
    inputs = processor(audio, sampling_rate=TARGET_SR, return_tensors="pt")
    input_values = inputs.input_values.to(device)

    _, logits = model(input_values)
    logits = logits.cpu().numpy()[0]

    # モデルの出力順は [arousal, dominance, valence]
    return {
        "arousal": float(logits[0]),
        "dominance": float(logits[1]),
        "valence": float(logits[2]),
    }


print("推論関数を定義しました。")

## ⑤ 一括推論の実行

In [ ]:
import glob
import pandas as pd

audio_files = sorted(
    glob.glob(os.path.join(AUDIO_DIR, "*.wav"))
    + glob.glob(os.path.join(AUDIO_DIR, "*.mp3"))
)

print(f"{len(audio_files)}件の音声ファイルが見つかりました。")

results = []
for path in audio_files:
    fname = os.path.basename(path)
    try:
        vad = predict_vad(path, model, processor, device)
        vad["file"] = fname
        results.append(vad)
        print(f"完了: {fname} -> {vad}")
    except Exception as e:
        print(f"エラー: {fname} -> {e}")

df = pd.DataFrame(results)[["file", "arousal", "dominance", "valence"]]
df

## ⑥ 結果の可視化

Valence(横軸) × Arousal(縦軸) の平面上に各音声をプロットします。
点の色の濃さはDominanceの値を表します。

In [ ]:
import matplotlib.pyplot as plt
import japanize_matplotlib  # 日本語ファイル名の文字化け防止

fig, ax = plt.subplots(figsize=(7, 7))

sc = ax.scatter(
    df["valence"],
    df["arousal"],
    c=df["dominance"],
    cmap="coolwarm",
    s=150,
    edgecolors="black",
)

for _, row in df.iterrows():
    ax.annotate(row["file"], (row["valence"], row["arousal"]), fontsize=8, xytext=(5, 5), textcoords="offset points")

ax.axvline(0.5, color="gray", linestyle="--", linewidth=0.8)
ax.axhline(0.5, color="gray", linestyle="--", linewidth=0.8)
ax.set_xlabel("Valence（ネガティブ ← → ポジティブ）")
ax.set_ylabel("Arousal（落ち着き ← → 興奮）")
ax.set_title("日本語音声のVAD分布（Before：ファインチューニングなし）")
plt.colorbar(sc, label="Dominance")
plt.tight_layout()
plt.savefig("/content/vad_before.png", dpi=150)
plt.show()

## ⑦ 結果の保存

Afterの結果と比較できるよう、CSVとして保存しておきます。

In [ ]:
OUTPUT_CSV = "/content/vad_results_before.csv"
df.to_csv(OUTPUT_CSV, index=False)
print(f"結果を保存しました: {OUTPUT_CSV}")

# 手元にダウンロードしたい場合
from google.colab import files as colab_files
colab_files.download(OUTPUT_CSV)

## 次のステップ

1. **結果の妥当性を目視確認**：怒りのテキストで読み上げた音声のArousalが高く出ているか、悲しみの音声のValenceが低く出ているか等をざっと確認
2. **JVNV等の本格データでの評価**：IEEE DataPortからJVNVをダウンロードし、方法Aで一括アップロードして同様に評価
3. **ファインチューニング（After）**：`wav2vec2`本体を凍結し、`RegressionHead`部分のみをJVNV等の訓練データで再学習
4. **Before/After比較**：同じ評価用音声に対するVAD値・分布の変化を比較

メモリ不足が発生した場合は、Colabの「ランタイム→ランタイムのタイプを変更」から有料プランのGPU（L4やA100等）に変更してください。